In [10]:
# Script: GenerarCsvBruto_Completo.py
import os
import xml.etree.ElementTree as ET
import pandas as pd
from tqdm import tqdm



def generar_csv_completo(CARPETA_TRAIN = "brain-1h-over-375day",ARCHIVO_SALIDA = "brain_bruto_1h.csv"):
    if not os.path.exists(CARPETA_TRAIN):
        print(f"❌ La carpeta '{CARPETA_TRAIN}' no existe.")
        return

    ficheros = sorted([f for f in os.listdir(CARPETA_TRAIN) if f.endswith(".xml")])
    print(f"📂 Encontrados {len(ficheros)} archivos XML. Iniciando proceso...")

    # Namespace del XML detectado anteriormente
    ns = {'ns': 'http://sndlib.zib.de/network'}
    
    datos_temporales = []

    # Bucle principal con barra de progreso
    for f in tqdm(ficheros, desc="Procesando XMLs"):
        ruta_completa = os.path.join(CARPETA_TRAIN, f)
        try:
            tree = ET.parse(ruta_completa)
            root = tree.getroot()
            
            # 1. Extraer Tiempo (tag <time> dentro de <meta>)
            tag_time = root.find(".//ns:time", ns)
            ts = tag_time.text if tag_time is not None else f.replace(".xml", "")
            
            fila = {'timestamp': ts}
            
            # 2. Extraer todas las demandas individuales
            for demand in root.findall(".//ns:demand", ns):
                src = demand.find("ns:source", ns).text
                dst = demand.find("ns:target", ns).text
                val_elem = demand.find("ns:demandValue", ns)
                
                if val_elem is not None:
                    nombre_enlace = f"{src}_{dst}"
                    fila[nombre_enlace] = float(val_elem.text)
            
            datos_temporales.append(fila)
            
        except Exception as e:
            # Si un archivo falla, continuamos con el siguiente
            continue

    print(f"\n📊 Creando DataFrame gigante (esto consumirá RAM)...")
    df = pd.DataFrame(datos_temporales)
    
    # 3. Limpieza y Ordenación
    print("🧹 Ordenando datos por tiempo...")
    df['timestamp'] = pd.to_datetime(df['timestamp'], format='%Y%m%d-%H%M', errors='coerce')
    df = df.sort_values('timestamp')
    
    # Reordenar columnas para que timestamp sea la primera
    cols = ['timestamp'] + sorted([c for c in df.columns if c != 'timestamp'])
    df = df[cols]
    
    print(f"💾 Guardando archivo CSV: {ARCHIVO_SALIDA}...")
    # Usamos index=False para no añadir la columna de IDs de pandas
    df.to_csv(ARCHIVO_SALIDA, index=False)
    
    print("-" * 40)
    print(f"✅ ¡PROCESO FINALIZADO!")
    print(f"Filas (instantes de tiempo): {len(df)}")
    print(f"Columnas (enlaces): {len(df.columns) - 1}")
    print(f"Archivo generado: {ARCHIVO_SALIDA}")
    print("-" * 40)

if __name__ == "__main__":
    generar_csv_completo(CARPETA_TRAIN = "brain-1h-over-375day",ARCHIVO_SALIDA = "brain_bruto_1h.csv")
    generar_csv_completo(CARPETA_TRAIN = "brain-1min-over-7days",ARCHIVO_SALIDA = "brain_bruto_1min.csv")

📂 Encontrados 8993 archivos XML. Iniciando proceso...


Procesando XMLs: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 8993/8993 [17:43<00:00,  8.45it/s]



📊 Creando DataFrame gigante (esto consumirá RAM)...
🧹 Ordenando datos por tiempo...
💾 Guardando archivo CSV: brain_bruto_1h.csv...
----------------------------------------
✅ ¡PROCESO FINALIZADO!
Filas (instantes de tiempo): 8993
Columnas (enlaces): 17663
Archivo generado: brain_bruto_1h.csv
----------------------------------------
📂 Encontrados 9723 archivos XML. Iniciando proceso...


Procesando XMLs: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9723/9723 [19:59<00:00,  8.10it/s]



📊 Creando DataFrame gigante (esto consumirá RAM)...
🧹 Ordenando datos por tiempo...
💾 Guardando archivo CSV: brain_bruto_1min.csv...
----------------------------------------
✅ ¡PROCESO FINALIZADO!
Filas (instantes de tiempo): 9723
Columnas (enlaces): 14311
Archivo generado: brain_bruto_1min.csv
----------------------------------------


In [11]:
import pandas as pd
import os

def agrupar_por_nodos(archivo_entrada, archivo_salida):
    if not os.path.exists(archivo_entrada):
        print(f"❌ No existe el archivo: {archivo_entrada}")
        return

    print(f"📖 Cargando cabecera de {archivo_entrada}...")
    # Leemos solo la primera fila para crear el mapa de nombres
    df_sample = pd.read_csv(archivo_entrada, nrows=1)
    todas_las_columnas = [c for c in df_sample.columns if c != 'timestamp']
    
    # Mapa: 'HU4_ADH63' -> 'HU_ADH'
    mapa_agrupacion = {}
    for col in todas_las_columnas:
        partes = col.split('_')
        if len(partes) == 2:
            src = ''.join([i for i in partes[0] if not i.isdigit()])
            dst = ''.join([i for i in partes[1] if not i.isdigit()])
            mapa_agrupacion[col] = f"{src}_{dst}"

    print(f"🧩 Mapeo completado. Enlaces core detectados: {len(set(mapa_agrupacion.values()))}")

    # Procesar por trozos para no reventar la RAM
    print("🚀 Iniciando agregación...")
    chunks = pd.read_csv(archivo_entrada, chunksize=1000)
    lista_df_agrupados = []

    for i, chunk in enumerate(chunks):
        ts = chunk['timestamp'].reset_index(drop=True)
        chunk_datos = chunk.drop(columns=['timestamp'])
        
        # --- SOLUCIÓN AL ERROR DE AXIS ---
        # Agrupamos usando el mapa sobre las columnas
        # T (transponemos), agrupamos por el mapa, sumamos y volvemos a transponer
        chunk_agrupado = chunk_datos.T.groupby(mapa_agrupacion).sum().T.reset_index(drop=True)
        
        # Volvemos a poner el timestamp
        chunk_agrupado.insert(0, 'timestamp', ts)
        lista_df_agrupados.append(chunk_agrupado)
        print(f"   Bloque {i+1} procesado...")

    print("🔗 Uniendo bloques...")
    df_final = pd.concat(lista_df_agrupados, ignore_index=True)

    print(f"💾 Guardando en {archivo_salida}...")
    df_final.to_csv(archivo_salida, index=False)
    
    print("-" * 30)
    print(f"✅ ¡TERMINADO!")
    print(f"Archivo: {archivo_salida}")
    print(f"Columnas finales: {len(df_final.columns)}")
    print("-" * 30)

if __name__ == "__main__":
    # Ejecutar para el de 1 hora
    agrupar_por_nodos("brain_bruto_1h.csv", "brain_CORE_1h.csv")
    agrupar_por_nodos("brain_bruto_1min.csv", "brain_CORE_1min.csv")

📖 Cargando cabecera de brain_bruto_1h.csv...
🧩 Mapeo completado. Enlaces core detectados: 81
🚀 Iniciando agregación...
   Bloque 1 procesado...
   Bloque 2 procesado...
   Bloque 3 procesado...
   Bloque 4 procesado...
   Bloque 5 procesado...
   Bloque 6 procesado...
   Bloque 7 procesado...
   Bloque 8 procesado...
   Bloque 9 procesado...
🔗 Uniendo bloques...
💾 Guardando en brain_CORE_1h.csv...
------------------------------
✅ ¡TERMINADO!
Archivo: brain_CORE_1h.csv
Columnas finales: 82
------------------------------
📖 Cargando cabecera de brain_bruto_1min.csv...
🧩 Mapeo completado. Enlaces core detectados: 81
🚀 Iniciando agregación...
   Bloque 1 procesado...
   Bloque 2 procesado...
   Bloque 3 procesado...
   Bloque 4 procesado...
   Bloque 5 procesado...
   Bloque 6 procesado...
   Bloque 7 procesado...
   Bloque 8 procesado...
   Bloque 9 procesado...
   Bloque 10 procesado...
🔗 Uniendo bloques...
💾 Guardando en brain_CORE_1min.csv...
------------------------------
✅ ¡TERMINADO!

In [13]:
import pandas as pd
import numpy as np

def analizar_archivo(ruta_csv):
    print(f"\n🔍 Analizando: {ruta_csv}")
    print("-" * 40)
    
    try:
        # Cargamos el archivo
        df = pd.read_csv(ruta_csv)
        
        # 1. Conteo total de NaNs
        total_nans = df.isna().sum().sum()
        
        # 2. Filas totalmente vacías
        filas_vacias = df.drop(columns=['timestamp']).isna().all(axis=1).sum()
        
        # 3. Columnas con NaNs
        columnas_con_nans = df.columns[df.isna().any()].tolist()
        
        print(f"✅ Total de celdas en el archivo: {df.size}")
        print(f"⚠️ Total de NaNs encontrados: {total_nans}")
        print(f"🚫 Filas completamente vacías: {filas_vacias}")
        print(f"📊 Número de columnas: {len(df.columns)}")
        print(f"📈 Número de filas: {len(df)}")

        if total_nans > 0:
            print("\n📍 Detalle de NaNs por columna (Top 10):")
            detalles = df[columnas_con_nans].isna().sum().sort_values(ascending=False).head(10)
            print(detalles)
            
            # Opción para limpiar en el momento
            print("\n💡 Sugerencia: Usa 'df.fillna(0)' si quieres convertir los NaNs en tráfico cero.")
        else:
            print("\n💎 ¡Perfecto! El archivo no tiene valores nulos.")

    except Exception as e:
        print(f"❌ Error al leer el archivo: {e}")

if __name__ == "__main__":
    # Comprueba el que acabamos de agrupar
    analizar_archivo("brain_CORE_1h.csv")
    analizar_archivo("brain_CORE_1min.csv")
    # También puedes comprobar el bruto si quieres
    # analizar_archivo("brain_bruto_1h.csv")


🔍 Analizando: brain_CORE_1h.csv
----------------------------------------
✅ Total de celdas en el archivo: 737426
⚠️ Total de NaNs encontrados: 0
🚫 Filas completamente vacías: 0
📊 Número de columnas: 82
📈 Número de filas: 8993

💎 ¡Perfecto! El archivo no tiene valores nulos.

🔍 Analizando: brain_CORE_1min.csv
----------------------------------------
✅ Total de celdas en el archivo: 797286
⚠️ Total de NaNs encontrados: 0
🚫 Filas completamente vacías: 0
📊 Número de columnas: 82
📈 Número de filas: 9723

💎 ¡Perfecto! El archivo no tiene valores nulos.


In [16]:
import pandas as pd
import numpy as np

def enriquecer_y_guardar(archivo_entrada, archivo_salida):
    print(f"📖 Leyendo {archivo_entrada}...")
    df = pd.read_csv(archivo_entrada)
    
    # Asegurar formato fecha
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    
    print("🛠️ Añadiendo características de calendario...")
    
    # 1. Componentes básicos
    df['hour'] = df['timestamp'].dt.hour
    df['day_of_week'] = df['timestamp'].dt.dayofweek # Lunes=0, Domingo=6
    
    # 2. Codificación Cíclica (Seno/Coseno)
    # Esto ayuda al modelo a entender que las 23:00 y las 00:00 están pegadas
    df['hour_sin'] = np.sin(2 * np.pi * df['hour']/24)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour']/24)
    
    # Esto ayuda a entender que el Domingo está pegado al Lunes
    df['day_sin'] = np.sin(2 * np.pi * df['day_of_week']/7)
    df['day_cos'] = np.cos(2 * np.pi * df['day_of_week']/7)
    
    # 3. Variable binaria de fin de semana
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    
    # 4. Limpieza: Rellenar posibles huecos con 0 (muy importante)
    df = df.fillna(0)
    
    print(f"💾 Guardando archivo final: {archivo_salida}...")
    df.to_csv(archivo_salida, index=False)
    
    print("-" * 30)
    print("✅ ¡ARCHIVO FINAL LISTO!")
    print(f"Nuevas columnas añadidas: hour_sin, hour_cos, day_sin, day_cos, is_weekend")
    print(f"Total de columnas: {len(df.columns)}")
    print("-" * 30)

if __name__ == "__main__":
    # Generamos los archivos finales enriquecidos
    enriquecer_y_guardar("brain_CORE_1h.csv", "brain_FINAL_1h.csv")
    enriquecer_y_guardar("brain_CORE_1min.csv", "brain_FINAL_1min.csv")

📖 Leyendo brain_CORE_1h.csv...
🛠️ Añadiendo características de calendario...
💾 Guardando archivo final: brain_FINAL_1h.csv...
------------------------------
✅ ¡ARCHIVO FINAL LISTO!
Nuevas columnas añadidas: hour_sin, hour_cos, day_sin, day_cos, is_weekend
Total de columnas: 89
------------------------------
📖 Leyendo brain_CORE_1min.csv...
🛠️ Añadiendo características de calendario...
💾 Guardando archivo final: brain_FINAL_1min.csv...
------------------------------
✅ ¡ARCHIVO FINAL LISTO!
Nuevas columnas añadidas: hour_sin, hour_cos, day_sin, day_cos, is_weekend
Total de columnas: 89
------------------------------


In [1]:
import pandas as pd
import numpy as np

def autopsia_csv(fichero):
    print(f"\n--- Autopsia de {fichero} ---")
    df = pd.read_csv(fichero)
    
    # 1. Buscar ceros reales
    total_ceros = (df == 0).sum().sum()
    print(f"Empty/Zero values: {total_ceros} ({total_ceros/df.size:.2%} del total)")
    
    # 2. Buscar outliers extremos (posibles errores de lectura)
    # Si el valor máximo es 1000 veces mayor que la mediana, hay algo raro.
    for col in df.select_dtypes(include=[np.number]).columns[:5]: # Solo las primeras 5 para no saturar
        mediana = df[col].median()
        maximo = df[col].max()
        if mediana > 0 and (maximo / mediana) > 1000:
            print(f"⚠️ Columna {col}: Máximo ({maximo}) es >1000x la mediana ({mediana}). Posible outlier.")

    # 3. Buscar columnas con varianza cero (datos muertos)
    constantes = [col for col in df.columns if df[col].nunique() <= 1]
    print(f"Columns with constant values: {len(constantes)}")
    if constantes:
        print(f"Ejemplos de columnas muertas: {constantes[:3]}")

autopsia_csv("brain_CORE_1h.csv")
autopsia_csv("brain_CORE_1min.csv")


--- Autopsia de brain_CORE_1h.csv ---
Empty/Zero values: 162 (0.02% del total)
⚠️ Columna ADH_ADH: Máximo (117010326.0) es >1000x la mediana (12873.0). Posible outlier.
Columns with constant values: 0

--- Autopsia de brain_CORE_1min.csv ---
Empty/Zero values: 0 (0.00% del total)
⚠️ Columna ADH_ADH: Máximo (42692498.0) es >1000x la mediana (1263.0). Posible outlier.
Columns with constant values: 0


In [1]:
# ============================================================
# MediaHoraria_desde1min (celda de Jupyter para PreProceDatos) — v1.0
# Calcula la media horaria REAL de cada enlace a partir de los datos de 1 min.
#   Convención BRAIN: la hora con etiqueta T es la media de los minutos (T-60, T]
#   (ej.: 15:01 … 16:00 → etiqueta 16:00).
# Salidas:
#   1) brain_MEDIA1h_desde1min.csv   → serie horaria (una fila por hora) +
#      n_minutos (cuántos minutos había en esa hora; 60 = hora completa)
#   2) brain_FINAL_1min_con_MEDIAH.csv → el CSV de 1 min con una columna
#      MEDIAH_<enlace> por enlace: la media de la hora a la que pertenece cada minuto
# Además compara la media de los minutos con brain_FINAL_1h.csv en las horas comunes.
# ============================================================
import numpy as np
import pandas as pd

ARCHIVO_1MIN    = "brain_FINAL_1min.csv"
ARCHIVO_1H      = "brain_FINAL_1h.csv"            # solo para comparar
SALIDA_HORARIA  = "brain_MEDIA1h_desde1min.csv"
SALIDA_MINUTOS  = "brain_FINAL_1min_con_MEDIAH.csv"
COLS_NO_ENLACE  = ["timestamp", "hour", "day_of_week",
                   "hour_sin", "hour_cos", "day_sin", "day_cos", "is_weekend"]

df = pd.read_csv(ARCHIVO_1MIN, parse_dates=["timestamp"]).sort_values("timestamp")
enlaces = [c for c in df.columns if c not in COLS_NO_ENLACE]
serie = df.set_index("timestamp")[enlaces]
print(f"📂 {ARCHIVO_1MIN}: {len(df)} minutos, {len(enlaces)} enlaces, "
      f"{serie.index.min()} → {serie.index.max()}")

# ── 1) Media horaria desde los minutos ─────────────────────
agr = serie.resample("1h", label="right", closed="right")
media_h = agr.mean()
media_h["n_minutos"] = agr.size()
media_h = media_h[media_h["n_minutos"] > 0]
media_h.reset_index().to_csv(SALIDA_HORARIA, index=False)

incompletas = media_h[media_h["n_minutos"] < 60]
print(f"💾 {SALIDA_HORARIA}: {len(media_h)} horas | "
      f"{len(incompletas)} horas incompletas (<60 min)")
if len(incompletas):
    print(incompletas["n_minutos"].describe().round(1).to_string())

# ── 2) Columna MEDIAH_<enlace> en el CSV de minutos ────────
#   Cada minuto t pertenece a la hora ceil(t) (15:01 … 16:00 → 16:00)
hora_de_cada_min = df["timestamp"].dt.ceil("h")
mediah = media_h[enlaces].reindex(hora_de_cada_min.values)
mediah.columns = [f"MEDIAH_{e}" for e in enlaces]
df_out = pd.concat([df.reset_index(drop=True), mediah.reset_index(drop=True)], axis=1)
df_out.to_csv(SALIDA_MINUTOS, index=False)
print(f"💾 {SALIDA_MINUTOS}: {df_out.shape[0]} filas, {df_out.shape[1]} columnas")

# ── 3) Comparación con el CSV horario original ─────────────
h = pd.read_csv(ARCHIVO_1H, parse_dates=["timestamp"]).set_index("timestamp")
comunes = media_h.index.intersection(h.index)
completas = media_h.loc[comunes, "n_minutos"] == 60
idx = comunes[completas.values]
dif = pd.Series({e: 100 * np.abs(media_h.loc[idx, e] - h.loc[idx, e]).sum()
                    / max(h.loc[idx, e].sum(), 1e-9) for e in enlaces})
print(f"\n🔍 Media de minutos vs {ARCHIVO_1H}: {len(idx)} horas completas en común")
print("   Diferencia relativa por enlace (%):")
print(dif.describe().round(2).to_string())
print("   Enlaces con más diferencia:")
print(dif.sort_values(ascending=False).head(10).round(2).to_string())

📂 brain_FINAL_1min.csv: 9723 minutos, 81 enlaces, 2013-03-08 14:52:00 → 2013-03-15 08:56:00
💾 brain_MEDIA1h_desde1min.csv: 163 horas | 3 horas incompletas (<60 min)
count     3.0
mean     41.0
std      28.6
min       8.0
25%      32.0
50%      56.0
75%      57.5
max      59.0
💾 brain_FINAL_1min_con_MEDIAH.csv: 9723 filas, 170 columnas

🔍 Media de minutos vs brain_FINAL_1h.csv: 160 horas completas en común
   Diferencia relativa por enlace (%):
count    81.00
mean     22.82
std      12.25
min       0.88
25%      15.47
50%      21.44
75%      26.60
max      78.33
   Enlaces con más diferencia:
HU_HTW       78.33
WIAS_WIAS    53.16
ADH_ADH      47.37
SPK_HTW      46.24
TU_TU        45.66
TU_HTW       44.90
UP_HU        43.25
WIAS_HU      42.35
HU_WIAS      37.04
UP_HTW       35.15
